# Phase 1 -- p1_5.1: Model C test predictions (tuned output settings)

Loads Model C's newest checkpoint, compares stronger repetition penalties
(1.3 / 1.4 / 1.5) on 300 dev examples (the first 100 are the same IDs used for
A, B and the earlier C run), keeps the best, and writes `submission.zip`.

**Setup:** GPU T4 x2, internet on. Inputs: `legallm-phase1-data`,
`legallm-tokenizers`, and `legallm-checkpoints-c` (latest version) -- or
C's notebook output (Add Input -> Notebook Output Files -> p1_07_model_C).
Secret on: `HF`. **Time:** ~15-20 min (GPU on).


In [ ]:
import json
import math
import shutil
import subprocess
import time
import zipfile
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint
from tqdm.auto import tqdm

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
KAGGLE_INPUT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/phase1_predict_c")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def find_input_dir(marker, label):
    hits = sorted(KAGGLE_INPUT.rglob(marker)) if KAGGLE_INPUT.exists() else []
    if not hits:
        raise FileNotFoundError(f"Could not find '{marker}'. Attach '{label}'.")
    return hits[0].parent


def step_of(path):
    return int(Path(path).stem.split("_step")[1].split("_tokens")[0])


def find_latest_checkpoint(prefix):
    pts = list(KAGGLE_INPUT.rglob(f"{prefix}_step*.pt")) if KAGGLE_INPUT.exists() else []
    if pts:
        return max(pts, key=step_of)
    best = None
    for zpath in (KAGGLE_INPUT.rglob("*.zip") if KAGGLE_INPUT.exists() else []):
        try:
            with zipfile.ZipFile(zpath) as zf:
                for name in zf.namelist():
                    base = Path(name).name
                    if base.startswith(f"{prefix}_step") and base.endswith(".pt"):
                        if best is None or step_of(base) > best[0]:
                            best = (step_of(base), zpath, name)
        except zipfile.BadZipFile:
            continue
    if best is None:
        return None
    _, zpath, name = best
    dest = OUTPUT_DIR / Path(name).name
    with zipfile.ZipFile(zpath) as zf, zf.open(name) as src, open(dest, "wb") as dst:
        shutil.copyfileobj(src, dst)
    return dest


INPUT_DATA_DIR = find_input_dir("dev_prompts.json", "legallm-phase1-data")
INPUT_TOKENIZER_DIR = find_input_dir("t2_unigram.model", "legallm-tokenizers")
CKPT_PATH = find_latest_checkpoint("C")          # main-run files only ("Csmoke_..." never match)
assert CKPT_PATH is not None, "No C_step*.pt found -- attach legallm-checkpoints-c (latest) or p1_07's output."
print("Checkpoint:", CKPT_PATH)

ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
MODEL_C_CONFIG = ckpt["config"]
print("Config:", MODEL_C_CONFIG, "| step", ckpt["step"], "| tokens", f"{ckpt['tokens_seen']:,}")
TRAIN_SEQ_LEN = 1024


In [ ]:
import sentencepiece as spm
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

sp = spm.SentencePieceProcessor()
sp.load(str(INPUT_TOKENIZER_DIR / "t2_unigram.model"))
EOS_ID = sp.eos_id()
BANNED_IN_OUTPUT = sorted({sp.pad_id(), sp.unk_id(), sp.bos_id(), sp.eos_id()} - {-1})


def encode(text):
    return sp.encode(text, out_type=int)


def decode(ids):
    return sp.decode(ids)


test_path = hf_hub_download("Exploration-Lab/CS-781-Capstone", "Phase_1_Pretraining/Test/test.json",
                            repo_type="dataset", token=UserSecretsClient().get_secret("HF"))
test_prompts = json.load(open(test_path))
test_ids = list(test_prompts)
dev_prompts = json.load(open(INPUT_DATA_DIR / "dev_prompts.json"))
dev_truth = json.load(open(INPUT_DATA_DIR / "dev_truth.json"))
dev_ids = sorted(set(dev_prompts) & set(dev_truth))[:300]      # first 100 = same IDs as A and B
print(f"Test prompts: {len(test_ids)} | dev examples: {len(dev_ids)}")


## Model C (identical code to p1_07) and its weights

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        return (x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight.float()).to(dtype)


def build_rope_cache(seq_len, d_head, theta=10000.0):
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_head, 2).float() / d_head))
    freqs = torch.outer(torch.arange(seq_len).float(), inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    """x [B,H,T,d]; cos/sin [B or 1, T, d/2]. Rotation in fp32."""
    x1, x2 = x[..., 0::2].float(), x[..., 1::2].float()
    cos, sin = cos[:, None].float(), sin[:, None].float()
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2).to(x.dtype)


def prefix_lm_mask(prefix_len, T, device):
    """Bool [B, 1, T, T]: key k visible to query q iff k <= q or k < prefix_len[b]."""
    idx = torch.arange(T, device=device)
    causal = idx[None, :, None] >= idx[None, None, :]
    in_prefix = idx[None, None, :] < prefix_len[:, None, None]
    return (causal | in_prefix)[:, None]


class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, h, kv, dh, b = cfg["d_model"], cfg["n_head"], cfg["n_kv_head"], cfg["d_head"], cfg["bias"]
        self.n_head, self.n_kv_head, self.d_head = h, kv, dh
        self.q_proj = nn.Linear(d, h * dh, bias=b)
        self.k_proj = nn.Linear(d, kv * dh, bias=b)
        self.v_proj = nn.Linear(d, kv * dh, bias=b)
        self.o_proj = nn.Linear(h * dh, d, bias=b)
        self.q_norm = RMSNorm(dh) if cfg["qk_norm"] else None
        self.k_norm = RMSNorm(dh) if cfg["qk_norm"] else None

    def forward(self, x, cos, sin, attn_mask, cache=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        if self.q_norm is not None:
            q, k = self.q_norm(q), self.k_norm(k)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if cache is not None:
            k, v = cache.update(k, v)
        n_rep = self.n_head // self.n_kv_head
        if n_rep > 1:
            k, v = k.repeat_interleave(n_rep, dim=1), v.repeat_interleave(n_rep, dim=1)
        if attn_mask is None:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=T > 1)
        else:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head))


class GeluMLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.up_proj = nn.Linear(cfg["d_model"], cfg["d_ff"], bias=cfg["bias"])
        self.down_proj = nn.Linear(cfg["d_ff"], cfg["d_model"], bias=cfg["bias"])

    def forward(self, x):
        return self.down_proj(F.gelu(self.up_proj(x), approximate="tanh"))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg["d_model"])
        self.attn = Attention(cfg)
        self.mlp_norm = RMSNorm(cfg["d_model"])
        self.mlp = GeluMLP(cfg)

    def _inner(self, x, cos, sin, attn_mask, cache):
        x = x + self.attn(self.attn_norm(x), cos, sin, attn_mask, cache)
        return x + self.mlp(self.mlp_norm(x))

    def forward(self, x, cos, sin, attn_mask, cache=None, use_checkpoint=False):
        if use_checkpoint and cache is None and self.training:
            return grad_checkpoint(self._inner, x, cos, sin, attn_mask, None, use_reentrant=False)
        return self._inner(x, cos, sin, attn_mask, cache)


class KVCache:
    def __init__(self, batch_size, n_kv_head, max_len, d_head, device, dtype):
        self.k = torch.zeros(batch_size, n_kv_head, max_len, d_head, device=device, dtype=dtype)
        self.v = torch.zeros_like(self.k)
        self.pos = 0

    def update(self, k, v):
        T = k.shape[2]
        self.k[:, :, self.pos:self.pos + T] = k
        self.v[:, :, self.pos:self.pos + T] = v
        self.pos += T
        return self.k[:, :, :self.pos], self.v[:, :, :self.pos]


class PrefixLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed = nn.Embedding(config["vocab_size"], config["d_model"])
        self.blocks = nn.ModuleList([Block(config) for _ in range(config["n_layer"])])
        self.norm_f = RMSNorm(config["d_model"])
        self.head = nn.Linear(config["d_model"], config["vocab_size"], bias=False)
        self.head.weight = self.embed.weight
        cos, sin = build_rope_cache(config["max_seq_len"], config["d_head"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config["n_layer"]))

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

    def encode(self, input_ids, attention_mask=None, mode="prefix", prefix_len=None):
        """Contract method: hidden states. mode 'causal' (prefix 0), 'prefix'
        (needs prefix_len [B]) or 'bidirectional' (whole input is prefix)."""
        B, T = input_ids.shape
        if mode == "causal" or prefix_len is None and mode == "prefix":
            prefix_len = torch.zeros(B, dtype=torch.long, device=input_ids.device)
        elif mode == "bidirectional":
            prefix_len = torch.full((B,), T, dtype=torch.long, device=input_ids.device)
        mask = prefix_lm_mask(prefix_len, T, input_ids.device) if bool((prefix_len > 0).any()) else None
        x = self.embed(input_ids)
        cos, sin = self.rope_cos[None, :T], self.rope_sin[None, :T]
        use_ckpt = self.training and USE_GRAD_CHECKPOINTING
        for blk in self.blocks:
            x = blk(x, cos, sin, mask, use_checkpoint=use_ckpt)
        return self.norm_f(x)

    def forward(self, ids, prefix_len=None, targets=None):
        """No targets -> logits. With targets -> [1, 3] = [loss sum, number of
        loss tokens, sum of logsumexp^2 (z-loss)] over positions that predict
        continuation tokens (t >= prefix_len - 1). Sums (not means) so
        DataParallel replicas combine exactly."""
        use_amp = self.training and ids.is_cuda
        if prefix_len is None:
            prefix_len = torch.zeros(ids.shape[0], dtype=torch.long, device=ids.device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = self.head(self.encode(ids, mode="prefix", prefix_len=prefix_len))
        if targets is None:
            return logits
        logits32 = logits.float()
        lse = torch.logsumexp(logits32, dim=-1)
        nll = lse - logits32.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
        pos = torch.arange(ids.shape[1], device=ids.device)[None, :]
        w = (pos >= (prefix_len[:, None] - 1)).float()
        return torch.stack([(nll * w).sum(), w.sum(), (lse.pow(2) * w).sum()]).unsqueeze(0)

    def step(self, ids, positions, caches, attn_mask):
        """Generation step with a KV cache. ids/positions [B, T]; attn_mask
        bool [B, 1, T, cache_len + T]."""
        x = self.embed(ids)
        cos, sin = self.rope_cos[positions], self.rope_sin[positions]
        for blk, cache in zip(self.blocks, caches):
            x = blk(x, cos, sin, attn_mask, cache=cache)
        return self.head(self.norm_f(x))

    def generate(self, input_ids, max_new_tokens, kv_cache=True, sampler="greedy", **kw):
        """Contract method (greedy, one prompt): the prompt is the prefix."""
        assert kv_cache and sampler == "greedy"
        toks = generate_batch(self, [input_ids[0].tolist()], max_new_tokens=max_new_tokens, **kw)[0]
        return torch.tensor([input_ids[0].tolist() + toks], device=input_ids.device)


@torch.no_grad()
def generate_batch(model, prompt_id_lists, max_new_tokens=288, max_context=None, rep_penalty=1.0,
                   no_repeat_ngram=0, target_words=None, banned=(), vocab_limit=None):
    """Greedy continuations for several prompts at once (left-padded). The
    prompt is the bidirectional prefix; continuation is causal. Optional:
    repetition penalty over generated tokens, n-gram blocking inside the
    continuation, stop once every row has > target_words words."""
    model.eval()
    device = next(model.parameters()).device
    dtype = next(model.parameters()).dtype
    cfg = model.config
    max_context = max_context or cfg["max_seq_len"]
    prompts = [p[-(max_context - max_new_tokens):] for p in prompt_id_lists]
    B, T0 = len(prompts), max(len(p) for p in prompts)
    pad = torch.tensor([T0 - len(p) for p in prompts], device=device)
    ids = torch.zeros(B, T0, dtype=torch.long, device=device)
    for b, p in enumerate(prompts):
        ids[b, T0 - len(p):] = torch.tensor(p, device=device)
    total = T0 + max_new_tokens
    caches = [KVCache(B, cfg["n_kv_head"], total, cfg["d_head"], device, dtype) for _ in range(cfg["n_layer"])]
    idx = torch.arange(total, device=device)
    key_ok = idx[None, :] >= pad[:, None]
    positions = (idx[None, :] - pad[:, None]).clamp(min=0)
    # Prefill: the prompt attends to itself in both directions (no padding keys);
    # every query may see itself so padded rows are never fully masked.
    mask = (key_ok[:, None, :T0] | torch.eye(T0, dtype=torch.bool, device=device)[None])[:, None]
    logits = model.step(ids, positions[:, :T0], caches, mask)[:, -1, :].float()
    generated = [[] for _ in range(B)]
    seen = [dict() for _ in range(B)]
    for t in range(max_new_tokens):
        if banned:
            logits[:, list(banned)] = -float("inf")
        if vocab_limit is not None:
            logits[:, vocab_limit:] = -float("inf")     # ids the tokenizer cannot decode
        if rep_penalty != 1.0 and t > 0:
            prev = torch.tensor(generated, device=device)
            sc = logits.gather(1, prev)
            logits.scatter_(1, prev, torch.where(sc > 0, sc / rep_penalty, sc * rep_penalty))
        if no_repeat_ngram and t >= no_repeat_ngram - 1:
            n = no_repeat_ngram
            for b in range(B):
                bad = seen[b].get(tuple(generated[b][-(n - 1):]) if n > 1 else ())
                if bad:
                    logits[b, list(bad)] = -float("inf")
        nxt = logits.argmax(dim=-1)
        for b, tok in enumerate(nxt.tolist()):
            generated[b].append(tok)
            if no_repeat_ngram and len(generated[b]) >= no_repeat_ngram:
                g = generated[b][-no_repeat_ngram:]
                seen[b].setdefault(tuple(g[:-1]), set()).add(g[-1])
        if t == max_new_tokens - 1:
            break
        if target_words and t >= 120 and t % 16 == 0 and all(
                len(decode(g).split()) > target_words for g in generated):
            break
        cur = T0 + t
        logits = model.step(nxt[:, None], positions[:, cur:cur + 1], caches,
                            key_ok[:, None, None, :cur + 1])[:, -1, :].float()
    return generated


def build_model(config):
    """Contract function."""
    return PrefixLM(config)


USE_GRAD_CHECKPOINTING = True


def _model_self_test():
    torch.manual_seed(0)
    tiny = dict(MODEL_C_CONFIG, vocab_size=101, d_model=64, n_layer=3, n_head=4, n_kv_head=2,
                d_head=16, d_ff=128, max_seq_len=64)
    m = build_model(tiny).to(DEVICE).eval()
    ids = torch.randint(4, 101, (2, 30), device=DEVICE)
    P = torch.tensor([12, 12], device=DEVICE)
    with torch.no_grad():
        full = m(ids, prefix_len=P)
        caches = [KVCache(2, 2, 30, 16, DEVICE, torch.float32) for _ in range(3)]
        pos = torch.arange(30, device=DEVICE)[None].expand(2, -1)
        parts = [m.step(ids[:, :12], pos[:, :12], caches, torch.ones(2, 1, 12, 12, dtype=torch.bool, device=DEVICE))]
        for t in range(12, 30):
            parts.append(m.step(ids[:, t:t + 1], pos[:, t:t + 1], caches,
                                torch.ones(2, 1, 1, t + 1, dtype=torch.bool, device=DEVICE)))
        diff = (full - torch.cat(parts, 1)).abs().max().item()
    assert diff < 1e-3, f"cached generation != full forward ({diff})"

    ids2 = ids.clone()
    ids2[:, 20] = (ids[:, 20] + 1) % 97 + 4          # change a continuation token
    ids3 = ids.clone()
    ids3[:, 5] = (ids[:, 5] + 1) % 97 + 4            # change a prompt token
    with torch.no_grad():
        leak = (m(ids2, prefix_len=P)[:, :20] - full[:, :20]).abs().max().item()
        sees_right = (m(ids3, prefix_len=P)[:, 2] - full[:, 2]).abs().max().item()
        causal_ok = (m(ids3)[:, 2] - m(ids)[:, 2]).abs().max().item()
    assert leak < 1e-5, f"future continuation leaked into earlier positions ({leak})"
    assert sees_right > 1e-5, "prompt tokens do not see later prompt tokens"
    assert causal_ok < 1e-5, "causal mode (prefix 0) sees the future"

    targets = torch.randint(4, 101, (2, 30), device=DEVICE)
    with torch.no_grad():
        s = m(ids, prefix_len=P, targets=targets)[0]
        ref = F.cross_entropy(full[:, 11:].reshape(-1, 101), targets[:, 11:].reshape(-1), reduction="sum")
    assert abs(s[0].item() - ref.item()) < 1e-3 and int(s[1].item()) == 2 * 19, "loss masking wrong"

    pa, pb = ids[0, :20].tolist(), ids[1, :9].tolist()
    both = generate_batch(m, [pa, pb], max_new_tokens=10)
    alone = [generate_batch(m, [pa], max_new_tokens=10)[0], generate_batch(m, [pb], max_new_tokens=10)[0]]
    assert both == alone, "batched (padded) generation differs from one-at-a-time"
    print(f"Self-test passed: cache-vs-full {diff:.1e}, no leak {leak:.1e}, prompt two-way ok, "
          f"loss mask ok, batched generation = single.")


_model_self_test()

model = build_model(MODEL_C_CONFIG)
model.load_state_dict(ckpt["model_state"])
del ckpt
model = model.to(DEVICE).eval()
print(f"Loaded Model C: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params")

# fp16 for speed, if it agrees with fp32 on a few dev prompts
check = [encode(dev_prompts[i]["text"]) for i in dev_ids[:4]]
KW = dict(max_new_tokens=64, max_context=TRAIN_SEQ_LEN, banned=BANNED_IN_OUTPUT, vocab_limit=sp.get_piece_size())
ref32 = generate_batch(model, check, **KW)
model = model.half()
model.rope_cos, model.rope_sin = model.rope_cos.float(), model.rope_sin.float()
out16 = generate_batch(model, check, **KW)
agree = sum(a == b for r, h in zip(ref32, out16) for a, b in zip(r, h)) / sum(len(r) for r in ref32)
if agree < 0.5:
    model = model.float()
print(f"fp16 vs fp32 agreement {agree:.0%} -> using {next(model.parameters()).dtype}")


## Compare output settings on 300 dev examples

The first run (p1_08) found `rep1.3_norepeat3` best, with BERTScore still
rising, so this run tests stronger penalties. The best BERTScore here is used
for the test set.


In [ ]:
try:
    import bert_score
except ImportError:
    subprocess.run(["pip", "install", "-q", "bert-score"], check=True)
    import bert_score


def lcs_length(a, b):
    if not a or not b:
        return 0
    if len(a) < len(b):
        a, b = b, a
    masks = {}
    for j, t in enumerate(a):
        masks[t] = masks.get(t, 0) | (1 << j)
    state = 0
    for t in b:
        m = masks.get(t, 0)
        x = m | state
        y = (state << 1) | 1
        state = x & ~(x - y)
    return state.bit_count() if hasattr(int, "bit_count") else bin(state).count("1")


def rouge_l_f1(p_text, r_text):
    pt, rt = p_text.split(), r_text.split()
    if not pt or not rt:
        return 0.0
    l = lcs_length(pt, rt)
    if not l:
        return 0.0
    pr, rc = l / len(pt), l / len(rt)
    return 2 * pr * rc / (pr + rc)


def ngrams(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)) if len(tokens) >= n else Counter()


def bleu4(preds, refs):
    clipped, totals, pl, rl = [0] * 4, [0] * 4, 0, 0
    for p, r in zip(preds, refs):
        pl += len(p)
        rl += len(r)
        for n in range(1, 5):
            pn, rn = ngrams(p, n), ngrams(r, n)
            totals[n - 1] += sum(pn.values())
            for g, c in pn.items():
                clipped[n - 1] += min(c, rn.get(g, 0))
    prec = [clipped[i] / totals[i] if totals[i] else 0.0 for i in range(4)]
    if any(x <= 0 for x in prec) or pl == 0:
        return 0.0
    return (1.0 if pl > rl else math.exp(1.0 - rl / pl)) * math.exp(sum(math.log(x) for x in prec) / 4)




def predict(prompt_dict, ids, desc, **settings):
    enc = {i: encode(prompt_dict[i]["text"]) for i in ids}
    order = sorted(ids, key=lambda i: len(enc[i]))
    out = {}
    for s in tqdm(range(0, len(order), 16), desc=desc, dynamic_ncols=True):
        chunk = order[s:s + 16]
        for i, toks in zip(chunk, generate_batch(model, [enc[i] for i in chunk], **settings)):
            words = decode(toks).split()[:128]
            out[i] = {"text": " ".join(words) if words else "."}
    return {i: out[i] for i in ids}


def score(pred, truth_ids):
    hyps = [pred[i]["text"].strip() for i in truth_ids]
    refs = [dev_truth[i]["text"].strip() for i in truth_ids]
    _, _, f1 = bert_score.score(hyps, refs, lang="en", batch_size=8, verbose=False,
                                device="cuda" if torch.cuda.is_available() else "cpu")
    return {"bertscore": float(f1.mean()),
            "rouge_l": sum(rouge_l_f1(h, r) for h, r in zip(hyps, refs)) / len(refs),
            "bleu4": bleu4([h.split() for h in hyps], [r.split() for r in refs])}


BASE = dict(max_new_tokens=288, max_context=TRAIN_SEQ_LEN, target_words=128,
            banned=BANNED_IN_OUTPUT, vocab_limit=sp.get_piece_size())
SETTINGS = {
    "rep1.3_norepeat3": dict(BASE, rep_penalty=1.3, no_repeat_ngram=3),
    "rep1.4_norepeat3": dict(BASE, rep_penalty=1.4, no_repeat_ngram=3),
    "rep1.5_norepeat3": dict(BASE, rep_penalty=1.5, no_repeat_ngram=3),
    "rep1.4_norepeat4": dict(BASE, rep_penalty=1.4, no_repeat_ngram=4),
}
results = {}
for name, st in SETTINGS.items():
    preds = predict(dev_prompts, dev_ids, f"dev:{name}", **st)
    results[name] = score(preds, dev_ids)
    r = results[name]
    print(f"{name:18s} BERTScore={r['bertscore']:.4f}  ROUGE-L={r['rouge_l']:.4f}  BLEU-4={r['bleu4']:.4f}")
    print("   sample:", preds[dev_ids[0]]["text"][:160], "...")
print("Reference (first 100 dev IDs): C rep1.3_norepeat3 = 0.8444 -> Codabench 0.8362 (dev-to-Codabench gap about -0.008)")
BEST = max(results, key=lambda k: (results[k]["bertscore"], results[k]["rouge_l"]))
print(f"\nBest setting: {BEST}")
json.dump(results, open(OUTPUT_DIR / "dev_settings_comparison.json", "w"), indent=2)


## Test predictions -> `submission.zip`

In [ ]:
t0 = time.time()
test_preds = predict(test_prompts, test_ids, "test", **SETTINGS[BEST])
assert set(test_preds) == set(test_ids) and all(v["text"].strip() for v in test_preds.values())
wc = [len(v["text"].split()) for v in test_preds.values()]
print(f"{len(test_preds)} predictions in {(time.time() - t0) / 60:.1f} min | words min {min(wc)}, "
      f"mean {sum(wc) / len(wc):.1f}, max {max(wc)}")
pred_path = OUTPUT_DIR / "prediction.json"
json.dump(test_preds, open(pred_path, "w", encoding="utf-8"), ensure_ascii=False)
zip_path = OUTPUT_DIR / "submission.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_path, arcname="prediction.json")
with zipfile.ZipFile(zip_path) as zf:
    assert zf.namelist() == ["prediction.json"] and set(json.loads(zf.read("prediction.json"))) == set(test_ids)
print(f"submission.zip ready (setting {BEST}, dev BERTScore {results[BEST]['bertscore']:.4f}).")
print("Download it from the Output tab: phase1_predict_c/submission.zip")
